# 🏥 Clinical Trial Dataset: Interactive EDA & Cleaning Pipeline

This notebook provides a step-by-step interactive workflow for analyzing and cleaning the clinical trial dataset. 
Each cell performs a specific check or transformation, allowing you to see the data quality at every stage.

In [ ]:
import pandas as pd
import os
import json

print("Libraries loaded successfully.")

## STEP 1: Data Ingestion

We load the raw dataset directly from the public S3 bucket.

In [ ]:
DB_PATH = "https://clinicaldata-765366202501-ap-southeast-2-an.s3.ap-southeast-2.amazonaws.com/healthcare_dataset.csv"
df = pd.read_csv(DB_PATH)

print(f"Loaded {len(df)} records.")
df.head()

## STEP 2: Exploratory Data Analysis (EDA)

### A. Missing Values Check

In [ ]:
missing_data = df.isnull().sum()
print("Missing values per column:")
print(missing_data[missing_data > 0] if missing_data.sum() > 0 else "No missing values found.")

### B. Duplicate Records Check

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Found {duplicate_count} duplicate rows.")
if duplicate_count > 0:
    display(df[df.duplicated()].head())

### C. Logical Anomalies (Age & Billing)

In [ ]:
invalid_ages = df[(df['Age'] < 0) | (df['Age'] > 120)]
negative_billing = df[df['Billing Amount'] < 0]

print(f"Invalid Ages (outside 0-120): {len(invalid_ages)}")
print(f"Negative Billing Records: {len(negative_billing)}")

if not negative_billing.empty:
    print("\nSample of billing anomalies:")
    display(negative_billing[['Name', 'Billing Amount']].head())

## STEP 3: Data Cleaning Operations

In [ ]:
# 1. Drop duplicates
df_clean = df.drop_duplicates().copy()

# 2. Fix billing (absolute value)
df_clean['Billing Amount'] = df_clean['Billing Amount'].abs()

# 3. Filter invalid ages
df_clean = df_clean[(df_clean['Age'] >= 0) & (df_clean['Age'] <= 120)]

print(f"Cleaning complete. Final dataset size: {len(df_clean)} records.")

## STEP 4: Standardizing Categorical Fields (Ontology Extraction)

We title-case medical terms and strip whitespace to ensure the Agent's FAISS index matches perfectly.

In [ ]:
categorical_columns = ['Gender', 'Blood Type', 'Medical Condition', 'Admission Type', 'Medication', 'Test Results']
ontology_dict = {}

for col in categorical_columns:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
    if col == 'Blood Type':
        df_clean[col] = df_clean[col].str.upper()
    
    unique_vals = sorted(df_clean[col].unique().tolist())
    ontology_dict[col] = unique_vals
    print(f"{col}: {len(unique_vals)} unique values")

print("\nStandardized categorical data sample:")
df_clean[categorical_columns].head()

## STEP 5: Exporting Cleaned Assets

We save the results to the project's `resources/` folder.

In [ ]:
# Determine project root
RESOURCES_DIR = os.path.join("..", "resources")
if not os.path.exists(RESOURCES_DIR):
    os.makedirs(RESOURCES_DIR)

# Export CSV
cleaned_csv_path = os.path.join(RESOURCES_DIR, "healthcare_dataset_cleaned.csv")
df_clean.to_csv(cleaned_csv_path, index=False)

# Export JSON Ontology
ontology_json_path = os.path.join(RESOURCES_DIR, "ontology_dictionary.json")
with open(ontology_json_path, "w") as f:
    json.dump(ontology_dict, f, indent=4)

print(f"✅ Success! Exported cleaned data and ontology to: {RESOURCES_DIR}")